# Creación de Modelo MLP para popularidad

In [ ]:
from src.utils.config import popularity
from src.utils.files import read_file

from sklearn.preprocessing import StandardScaler, PowerTransformer, MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.inspection import permutation_importance
from sklearn.neural_network import MLPRegressor
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from umap import UMAP
from math import sqrt

import matplotlib.pyplot as plt

from pandas import DataFrame, concat, Series
from numpy import vstack

In [ ]:
df = read_file(popularity)
df.head(2)

In [ ]:
df.columns

In [ ]:
# Para ver la distribución de distintas variables
df['total_games_by_publisher'].astype(float).plot(kind='hist')

In [ ]:
def _preprocess_train(df):
    """Función para transformar los datos para realiza MLP

    Args:
        df (DataFrame): Datos iniciales del modelo que van a ser procesados.
    
    Returns:
        DataFrame: Datos que contienen las variables regresoras.
        DataFrame: Datos que contienen las variables respuesta.
    """
    df = df.reset_index(drop=True)
    df = df.drop(columns=['price_range', 'id', 'name', 'v_resnet', 'v_convnext', 'yt_score'])

    # Separación de DataFrames en diferentes tipos de variables
    y = DataFrame(df['recomendaciones_totales'])
    X_num_log = df[['num_languages', 'total_games_by_publisher', 'total_games_by_developer', 'price_overview', 'description_len']]
    X_num_minmax = df[['release_year']] # Fechas
    X_num_std = df[['brillo']]
    X_youtube = df[[c for c in df.columns if 'video_statistics' in c]]
    X_trans = df[['Action', 'Adventure', 'Casual', 'Early Access', 'Free To Play', 'Indie', 'RPG', 'Simulation', 'Strategy', 'Co-op', 
                  'Custom Volume Controls', 'Family Sharing', 'Full controller support', 'Multi-player', 'Online Co-op', 'Online PvP', 
                  'Partial Controller Support', 'Playable without Timed Input', 'PvP', 'Remote Play Together', 'Shared/Split Screen', 
                  'Single-player', 'Steam Achievements', 'Steam Cloud', 'Steam Leaderboards', 'Steam Trading Cards']]

    # Transformación de variables
    pt1 = PowerTransformer(method='yeo-johnson')
    pt2 = PowerTransformer(method='yeo-johnson')
    y_trans = pt1.fit_transform(y)
    X_num_log_trans = pt2.fit_transform(X_num_log)

    std = StandardScaler()
    X_num_std_trans = std.fit_transform(X_num_std)

    mm = MinMaxScaler()
    X_num_minmax_trans = mm.fit_transform(X_num_minmax)

    pty = PowerTransformer(method='yeo-johnson')
    X_youtube_log = pty.fit_transform(X_youtube)
    pca = PCA(n_components=0.95)
    youtube_pca_trans = pca.fit_transform(X_youtube_log)

    # Datos a dataframes
    df_y_trans = DataFrame(y_trans, columns = pt1.get_feature_names_out())
    df_num_log_trans = DataFrame(X_num_log_trans, columns = pt2.get_feature_names_out())
    df_num_std_trans = DataFrame(X_num_std_trans, columns = std.get_feature_names_out())
    df_minmax_trans = DataFrame(X_num_minmax_trans, columns = mm.get_feature_names_out())
    df_youtube_pca_trans = DataFrame(youtube_pca_trans, columns = pca.get_feature_names_out())

    # Unificar datos transformados
    df1 = concat([df_num_log_trans, df_num_std_trans], axis=1)
    df2 = concat([df1, X_trans], axis=1)
    df3 = concat([df2, df_minmax_trans], axis=1)
    df4 = concat([df3, df_youtube_pca_trans], axis=1)

    # Aplicamos UMAP para reducir la dimensionalidad de los vectores de imágenes
    clip_matrix = vstack(df['v_clip'].values)
    umap = UMAP(n_components=16, random_state=42) 
    clip_reduced = umap.fit_transform(clip_matrix)
    
    for i in range(16):
        df4[f'clip_umap_{i}'] = clip_reduced[:, i]
    
    # Transformers usados para la transformación de los modelos
    transformers = {
        'pt1': pt1, 'pt2': pt2, 'std': std, 'mm': mm, 
        'pty': pty, 'pca': pca, 'umap': umap
    }

    return df4, df_y_trans, transformers

In [ ]:
def _preprocess_test(df, transformers):
    df = df.reset_index(drop=True)
    df = df.drop(columns=['price_range', 'id', 'name', 'v_resnet', 'v_convnext', 'yt_score'])

    # Separación de DataFrames
    y = DataFrame(df['recomendaciones_totales'])
    X_num_log = df[['num_languages', 'total_games_by_publisher', 'total_games_by_developer', 'price_overview', 'description_len']]
    X_num_minmax = df[['release_year']]
    X_num_std = df[['brillo']]
    X_youtube = df[[c for c in df.columns if 'video_statistics' in c]]
    X_trans = df[['Action', 'Adventure', 'Casual', 'Early Access', 'Free To Play', 'Indie', 'RPG', 'Simulation', 'Strategy', 'Co-op', 
                  'Custom Volume Controls', 'Family Sharing', 'Full controller support', 'Multi-player', 'Online Co-op', 'Online PvP', 
                  'Partial Controller Support', 'Playable without Timed Input', 'PvP', 'Remote Play Together', 'Shared/Split Screen', 
                  'Single-player', 'Steam Achievements', 'Steam Cloud', 'Steam Leaderboards', 'Steam Trading Cards']]

    # Transformación de variables usando transform()
    pt1 = transformers['pt1']
    pt2 = transformers['pt2']
    y_trans = pt1.transform(y)
    X_num_log_trans = pt2.transform(X_num_log)

    std = transformers['std']
    X_num_std_trans = std.transform(X_num_std)

    mm = transformers['mm']
    X_num_minmax_trans = mm.transform(X_num_minmax)

    pty = transformers['pty']
    X_youtube_log = pty.transform(X_youtube)
    pca = transformers['pca']
    youtube_pca_trans = pca.transform(X_youtube_log)

    # Datos a dataframes
    df_y_trans = DataFrame(y_trans, columns = pt1.get_feature_names_out())
    df_num_log_trans = DataFrame(X_num_log_trans, columns = pt2.get_feature_names_out())
    df_num_std_trans = DataFrame(X_num_std_trans, columns = std.get_feature_names_out())
    df_minmax_trans = DataFrame(X_num_minmax_trans, columns = mm.get_feature_names_out())
    df_youtube_pca_trans = DataFrame(youtube_pca_trans, columns = pca.get_feature_names_out())

    # Unificar datos transformados
    df1 = concat([df_num_log_trans, df_num_std_trans], axis=1)
    df2 = concat([df1, X_trans], axis=1)
    df3 = concat([df2, df_minmax_trans], axis=1)
    df4 = concat([df3, df_youtube_pca_trans], axis=1)

    # Aplicamos UMAP usando la matriz generada durante el entrenamiento
    clip_matrix = vstack(df['v_clip'].values)
    umap = transformers['umap']
    clip_reduced = umap.transform(clip_matrix)
    
    for i in range(16):
        df4[f'clip_umap_{i}'] = clip_reduced[:, i] 
    
    return df4, df_y_trans

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.3, random_state=42)

# Preprocesamiento
X_train, y_train, transformers = _preprocess_train(df_train)
X_test, y_test = _preprocess_test(df_test, transformers)

In [ ]:
# Modelo base para medir cuánto baja el rendimiento del mismo al quitar variables de forma individual
mlp_base = MLPRegressor(hidden_layer_sizes=(64,32), max_iter=5000, random_state=42)
mlp_base.fit(X_train, y_train.values.flatten())

test_importancia_variables = permutation_importance(
    mlp_base,
    X_test,
    y_test,
    random_state=42,
    n_jobs=-1,
    n_repeats=10
)

serie_ordenada_importancia = Series(test_importancia_variables.importances_mean, index=X_train.columns).sort_values(ascending=False)
serie_ordenada_importancia

Vamos a eliminar las variables menos útiles para el modelo:

clip_umap_4                     0.000395

clip_umap_6                     0.000298

clip_umap_15                    0.000207

clip_umap_7                    -0.000134

Single-player                  -0.000447

In [ ]:
X_train = X_train.drop(columns=['clip_umap_4','clip_umap_6','clip_umap_15','clip_umap_7','Single-player'])
X_test = X_test.drop(columns=['clip_umap_4','clip_umap_6','clip_umap_15','clip_umap_7','Single-player'])

In [ ]:
param_grid = {
    'hidden_layer_sizes': [(64,32), (124,), (80,60), (100,64)],
    'activation': ['relu', 'tanh'],
    'alpha': [0.0001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.01]
}

grid = GridSearchCV(MLPRegressor(max_iter=5000, random_state=42), param_grid=param_grid, cv=5, n_jobs=-1)
grid.fit(X_train, y_train.values.flatten())

In [ ]:
mlp_mejor_modelo = grid.best_estimator_
params_mejor_modelo = grid.best_params_
print(f'Los parámetros del mejor modelo son:\n{params_mejor_modelo}')

In [ ]:
y_pred = mlp_mejor_modelo.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

metricas = {
    "final_test_mae": mae,
    "final_test_rmse": rmse,
    "final_test_r2": r2,
}

print(metricas)